# 02. Bronze Ingestion Layer (Auto Loader)

Ingests raw Parquet files from Unity Catalog landing volumes into managed Delta Lake tables:
`bronze_prod.ecommerce.*`

### Ingestion Highlights:
- **Auto Loader (`cloudFiles`)**: Incremental ingestion with schema evolution and inference.
- **Rescued Data Tracking**: Malformed or unexpected columns are safely captured in `_rescued_data`.
- **Streaming Checkpointing**: Reliable state tracking using Unity Catalog volume checkpoints.
- **Trigger AvailableNow**: Production batch streaming pattern (`.trigger(availableNow=True)`).


In [0]:
%run ./01_Config


In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

dbutils.widgets.dropdown("p_dataset", "all", ["all", "orders", "products", "customers", "region"], "Dataset to Ingest")
p_dataset = dbutils.widgets.get("p_dataset")
print(f"[BRONZE] Ingesting dataset selection: '{p_dataset}'")


In [0]:
def ingest_dataset_auto_loader(dataset_name):
    """Reusable Auto Loader streaming ingestion pipeline for raw datasets into Bronze managed tables."""
    source_dir = f"{RAW_LANDING_VOLUME}/{dataset_name}"
    schema_dir = f"{CHECKPOINT_VOLUME}/{dataset_name}/schema"
    stream_checkpoint = f"{CHECKPOINT_VOLUME}/{dataset_name}/stream"
    target_table = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{dataset_name}"
    
    print(f"\n[AUTO LOADER] Starting ingestion for: {dataset_name}")
    print(f"  Source:     {source_dir}")
    print(f"  Target:     {target_table}")
    
    # 1. Read Stream with Auto Loader (cloudFiles)
    df_stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", schema_dir)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_dir)
    )
    
    # 2. Add Ingestion Audit Metadata
    df_stream_audited = df_stream.withColumn("_ingest_timestamp", current_timestamp())
    
    # 3. Write Stream to Managed Bronze Delta Table
    query = (
        df_stream_audited.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", stream_checkpoint)
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    
    query.awaitTermination()
    
    row_count = spark.table(target_table).count()
    print(f"✓ Ingestion completed: {target_table} now has {row_count:,} rows.")
    return row_count


In [0]:
# Execute Ingestion Pipeline
datasets_to_run = ["region", "products", "customers", "orders"] if p_dataset == "all" else [p_dataset]

ingest_results = []
for ds in datasets_to_run:
    count = ingest_dataset_auto_loader(ds)
    ingest_results.append((ds, count, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{ds}"))

print("\n[BRONZE] All ingestion tasks completed successfully.")


### Bronze Verification & Sample Previews


In [0]:
# Verification & Rescued Data Check
for ds, cnt, tbl in ingest_results:
    df_check = spark.table(tbl)
    has_rescued = "_rescued_data" in df_check.columns
    print(f"Table: {tbl} | Rows: {cnt:,} | Tracks _rescued_data: {has_rescued}")
    display(df_check.limit(3))
